# Job Data Collection with APIs

## What this notebook is
This is my **clean practice version** of an IBM Data Analyst course lab on collecting job data with APIs.

## Purpose
I used this notebook to review the main idea of the lab in a more structured way:
- make a simple API request
- read JSON data
- count job postings by technology
- save the results to Excel

## Note
This is a **personal practice adaptation** based on course learning. It is included to show my review process and understanding, not to claim authorship of the original course lab.


## Section 1 — Import the libraries

Here I import:
- `requests` to get data from an API
- `pandas` to organize the results in a table and export them to Excel


In [1]:
import requests
import pandas as pd


## Section 2 — Warm-up example: astronaut data API

Before working with job data, I use a small API example to review the basic flow:
1. send a GET request
2. convert the response to JSON
3. inspect the returned data


In [2]:
api_url = "http://api.open-notify.org/astros.json"
response = requests.get(api_url)

if response.ok:
    data = response.json()
    print(data)


{'people': [{'craft': 'ISS', 'name': 'Oleg Kononenko'}, {'craft': 'ISS', 'name': 'Nikolai Chub'}, {'craft': 'ISS', 'name': 'Tracy Caldwell Dyson'}, {'craft': 'ISS', 'name': 'Matthew Dominick'}, {'craft': 'ISS', 'name': 'Michael Barratt'}, {'craft': 'ISS', 'name': 'Jeanette Epps'}, {'craft': 'ISS', 'name': 'Alexander Grebenkin'}, {'craft': 'ISS', 'name': 'Butch Wilmore'}, {'craft': 'ISS', 'name': 'Sunita Williams'}, {'craft': 'Tiangong', 'name': 'Li Guangsu'}, {'craft': 'Tiangong', 'name': 'Li Cong'}, {'craft': 'Tiangong', 'name': 'Ye Guangfu'}], 'number': 12, 'message': 'success'}


### Check how many astronauts are currently on the ISS

`data.get('number')` reads the value stored under the `number` key.


In [3]:
print(data.get("number"))


12


### Print the astronaut names

Here I get the list stored under `people`, then loop through it and print each astronaut name.


In [4]:
astronauts = data.get("people")
print("There are {} astronauts on ISS".format(len(astronauts)))
print("And their names are:")
for astronaut in astronauts:
    print(astronaut.get("name"))


There are 12 astronauts on ISS
And their names are:
Oleg Kononenko
Nikolai Chub
Tracy Caldwell Dyson
Matthew Dominick
Michael Barratt
Jeanette Epps
Alexander Grebenkin
Butch Wilmore
Sunita Williams
Li Guangsu
Li Cong
Ye Guangfu


## Section 3 — Load the job data from the API source

Now I move to the real lab task.  
I request the job dataset and convert the response from JSON into a Python object.


In [5]:
api_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DA0321EN-SkillsNetwork/labs/module%201/Accessing%20Data%20Using%20APIs/jobs.json"

response = requests.get(api_url)
jobs_data = response.json()

print(type(jobs_data), "items:", len(jobs_data))
print(jobs_data[0])


<class 'list'> items: 27005
{'Id': 0, 'Job Title': 'Digital Media Planner', 'Job Experience Required': '5 - 10 yrs', 'Key Skills': 'Media Planning| Digital Media', 'Role Category': 'Advertising', 'Location': 'Los Angeles', 'Functional Area': 'Marketing , Advertising , MR , PR , Media Planning', 'Industry': 'Advertising, PR, MR, Event Management', 'Role': 'Media Planning Executive/Manager'}


## Section 4 — Create a function to count jobs by technology

This function checks each job record and counts how many postings mention a given technology in the **Key Skills** field.


In [6]:
def get_number_of_jobs_T(technology):
    count = 0

    for job in jobs_data:
        skills = job.get("Key Skills", "")
        if skills and technology.lower() in skills.lower():
            count += 1

    return technology, count


### Test the function with one example
This is a quick check to confirm that the function works.


In [7]:
get_number_of_jobs_T("Python")


('Python', 1173)

## Section 5 — Optional: create a function to count jobs by location

The original lab also refers to locations.  
This function uses the same logic, but checks the **Location** field instead of **Key Skills**.


In [8]:
def get_number_of_jobs_L(location):
    count = 0

    for job in jobs_data:
        job_location = job.get("Location", "")
        if job_location and location.lower() in job_location.lower():
            count += 1

    return location, count


### Test the location function


In [9]:
get_number_of_jobs_L("Los Angeles")


('Los Angeles', 640)

## Section 6 — Count postings for a list of technologies

Instead of checking one technology at a time, I store the technologies in a list and loop through them.


In [10]:
technologies = [
    "C", "C#", "C++", "Java", "JavaScript", "Python", "Scala",
    "Oracle", "SQL Server", "MySQL Server", "PostgreSQL", "MongoDB"
]


## Section 7 — Build the results table

Here I:
1. create an empty list
2. loop through the technologies
3. store each `(technology, count)` result
4. turn the final results into a pandas DataFrame


In [11]:
results = []

for tech in technologies:
    results.append(get_number_of_jobs_T(tech))

df = pd.DataFrame(results, columns=["Technology", "Number of Jobs"])
df


,Technology,Number of Jobs
0,C,25114
1,C#,526
2,C++,506
3,Java,3428
4,JavaScript,2248
5,Python,1173
6,Scala,138
7,Oracle,899
8,SQL Server,423
9,MySQL Server,0


## Section 8 — Export the results to Excel

This saves the final table as **job-postings.xlsx** so it can be used later in analysis or visualization work.


In [12]:
df.to_excel("job-postings.xlsx", index=False)


## Final note

This notebook shows a simple end-to-end beginner workflow:
- get data from an API source
- read JSON data
- count records based on a condition
- organize the results in a table
- export the output to Excel
